In [3]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

API_KEY = os.getenv("nvidiaapi_key")
MODEL = "openai/gpt-oss-20b"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def send_email(to: str, content: str) -> str:
    """이메일을 전송합니다."""

    print("받는 사람 : ", to)
    print("내용 : ", content)

    return f"{to}에게 이메일을 성공적으로 보냈습니다."


checkpointer = InMemorySaver()

hitl = HumanInTheLoopMiddleware(interrupt_on={"send_email": True})

agent = create_agent(
    model=llm, tools=[send_email], middleware=[hitl], checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "email_test_001"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.",
            }
        ]
    },
    config=config,
)

interrupts = result.get("__interrupt__")

print("interrupts")

decision = input("\n이메일을 보내겠습니까? (y = 승인 / n = 거절) : ")

if decision.lower() == "y":
    resume_command = Command(resume={"decisions": [{"type": "approve"}]})
else:
    resume_command = Command(
        resume={
            "decisions": [
                {"type": "reject", "message": "사용자가 이메일 발송을 거절했습니다."}
            ]
        }
    )

result = agent.invoke(resume_command, config=config)


for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

interrupts
받는 사람 :  kim.chesu@example.com
내용 :  김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.
HumanMessage
김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.
AIMessage

ToolMessage
kim.chesu@example.com에게 이메일을 성공적으로 보냈습니다.
AIMessage
이메일을 보내드렸습니다! 혹시 다른 요청이 있으신가요?


In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

load_dotenv(override=True)

# 1. 모델 준비
model = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
    model="openai/gpt-oss-20b",
)

# 2. 이메일을 가려주는 미들웨어를 하나만 붙인 에이전트
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="mask", apply_to_input=True),
    ],
)

# 3. 실행
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "제 이메일은 test@example.com 입니다."}
    ]
})

print(result["messages"][0].content)   # 가려진 입력 확인
print(result["messages"][-1].content)  # 에이전트 응답 확인

제 이메일은 test@****.com 입니다.
이메일 주소를 알려주셔서 감사합니다. 더 필요한 도움이나 질문이 있으면 언제든 말씀해 주세요!


In [9]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불규정
        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
                단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.
        
                배송 규정
        
                상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
                """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

0 : 환불규정
        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
                단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

                배송 규정

                상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


In [4]:
import os 
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

API_KEY = os.getenv("nvidiaapi_key")
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

embedding = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("documents : ", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunk = splitter.split_documents(documents)

print("chunk : ", len(chunks))

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

print("FAISS 생성완료")

query = "펀드가 무엇인가요?"

result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print("결과 : ", i)
    print("페이지 : ", doc.metadata.get("page"))
    print()
    print(doc.page_content)
    print()

c:\sk-encoa\llm_workspace\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


ValueError: File path ./rag_data/금융투자협회_투자길라잡이_2018.pdf is not a valid file or url